**Live demo:** 


**Name : Wajahat Nazir**  
**Roll no : sp24-bse-122**

# Heart Disease Prediction System
AIC354 Assignment 2 · UCI Heart Disease (Cleveland)

**Selected 4 input features:** `cp` (chest pain type), `thalach` (max heart rate), `exang` (exercise-induced angina), `oldpeak` (ST depression).
**Target:** `target` (0 = no disease, 1 = disease).

In [ ]:
import pandas as pd, numpy as np, joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## Phase 1: Data collection

In [ ]:
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
cols = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']
df = pd.read_csv(URL, names=cols, na_values='?')   # '?' marks missing values
df.to_csv('heart.csv', index=False)               # saved for submission
df.head()

## Data understanding

In [ ]:
print(df.shape)
df.info()
df.describe()

In [ ]:
print(df['target'].value_counts().sort_index())   # 0-4 severity levels
df[['cp','thalach','exang','oldpeak']].hist(figsize=(9,6)); plt.show()

## Data preprocessing
Keep only the 4 chosen features, check missing values, and make the target binary.

In [ ]:
data = df[['cp','thalach','exang','oldpeak','target']].copy()
print(data.isna().sum())
data = data.dropna()
data['target'] = (data['target'] > 0).astype(int)   # 0 = healthy, 1 = disease
data['cp'] = data['cp'].map({1:'Typical angina', 2:'Atypical angina', 3:'Non-anginal pain', 4:'No symptoms'})
data['exang'] = data['exang'].map({0:'No', 1:'Yes'})
data.head()

## Label encoding
The two text columns (`cp`, `exang`) are label-encoded inside the pipeline, so the saved model accepts raw text.

In [ ]:
X, y = data[['cp','thalach','exang','oldpeak']], data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

CP = ['Typical angina','Atypical angina','Non-anginal pain','No symptoms']
encode = ColumnTransformer([('label', OrdinalEncoder(categories=[CP, ['No','Yes']]), ['cp','exang'])],
                           remainder='passthrough')
encode.fit_transform(X_train)[:5]

## Phase 2 and 3: Model selection and training

In [ ]:
models = {'Logistic Regression': LogisticRegression(max_iter=1000),
          'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)}
fitted = {}
for name, m in models.items():
    p = Pipeline([('encode', encode), ('model', m)]).fit(X_train, y_train)
    fitted[name] = p
    print(name, 'test accuracy:', round(accuracy_score(y_test, p.predict(X_test)), 3))

## Phase 4: Evaluation

In [ ]:
best = max(fitted, key=lambda n: accuracy_score(y_test, fitted[n].predict(X_test)))
print('Best model:', best)
pred = fitted[best].predict(X_test)
print(classification_report(y_test, pred, target_names=['No disease','Disease']))
ConfusionMatrixDisplay.from_predictions(y_test, pred); plt.show()

## Save the model and test a prediction

In [ ]:
joblib.dump(fitted[best], 'heart_model.pkl')
sample = pd.DataFrame([{'cp':'Typical angina','thalach':120,'exang':'Yes','oldpeak':2.5}])
print('Prediction:', joblib.load('heart_model.pkl').predict(sample)[0])